In [ ]:
!pip install pandas pandera

In [ ]:
import urllib.request
import pandas as pd

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ops4life/mlops-get-started/main/datasets/HR-Employee-Attrition.csv",
    "HR-Employee-Attrition.csv"
)
# Run ingestion step to produce raw_ingested.csv
df_raw = pd.read_csv("HR-Employee-Attrition.csv")
df_raw.to_csv("raw_ingested.csv", index=False)
print("Setup complete: raw_ingested.csv ready.")

In [ ]:
import pandera.pandas as pa
from pandera import Column, Check
import pandas as pd

# 1. Define the schema
employee_schema = pa.DataFrameSchema(
    {
        "Age": Column(int, Check.between(18, 60), nullable=False),
        "Attrition": Column(str, Check.isin(["Yes", "No"]), nullable=False),
        "BusinessTravel": Column(str, Check.isin(["Non-Travel", "Travel_Rarely", "Travel_Frequently"]), nullable=False),
        "DailyRate": Column(int, Check.gt(0), nullable=False),
        "Department": Column(str, nullable=False),
        "DistanceFromHome": Column(int, Check.ge(1), nullable=False),
        "Education": Column(int, Check.between(1, 5), nullable=False),
        "EducationField": Column(str, nullable=False),
        "EmployeeNumber": Column(int, unique=True, nullable=False),
        "EnvironmentSatisfaction": Column(int, Check.between(1, 4), nullable=False),
        "Gender": Column(str, Check.isin(["Male", "Female"]), nullable=False),
        "HourlyRate": Column(int, Check.gt(0), nullable=False),
        "JobInvolvement": Column(int, Check.between(1, 4), nullable=False),
        "JobLevel": Column(int, Check.between(1, 5), nullable=False),
        "JobRole": Column(str, nullable=False),
        "JobSatisfaction": Column(int, Check.between(1, 4), nullable=False),
        "MaritalStatus": Column(str, Check.isin(["Single", "Married", "Divorced"]), nullable=False),
        "MonthlyIncome": Column(int, Check.gt(0), nullable=False),
        "OverTime": Column(str, Check.isin(["Yes", "No"]), nullable=False),
        "PerformanceRating": Column(int, Check.between(1, 4), nullable=False),
        "TotalWorkingYears": Column(int, Check.ge(0), nullable=False),
        "WorkLifeBalance": Column(int, Check.between(1, 4), nullable=False),
        "YearsAtCompany": Column(int, Check.ge(0), nullable=False),
        "YearsInCurrentRole": Column(int, Check.ge(0), nullable=False),
        "YearsSinceLastPromotion": Column(int, Check.ge(0), nullable=False),
    },
    strict=False,
    coerce=True,
)

def validate_data(df):
    try:
        validate_df = employee_schema(df, lazy=True)
        print("Data validation successful.")
        return validate_df

    except pa.errors.SchemaErrors as e:
        print("Data validation errors found:")
        print(e.failure_cases)
        return None


df = pd.read_csv("raw_ingested.csv")
validated_df = validate_data(df)
validated_df.to_csv("validated.csv", index=False)